# 13. Predict from the own leg, then test what the other leg adds

Does opposite-leg information improve a competent local predictor beyond the benefit of extra model capacity? We will examine the fitted models on the same coordinate target and learn why masking structure must be tested without changing the evaluation question.

## Run this notebook independently

From the repository root, install the shared environment and start Jupyter:

```bash
uv sync --project experiments/aaai
AAAI_ITERATION02_RUN_DIR=/absolute/path/to/experiments/aaai/artifacts/iteration02/PROTOCOL_HASH \
  uv run --project experiments/aaai jupyter lab experiments/aaai
```

Replace `PROTOCOL_HASH` with the exact completed run you intend to inspect. The setup displays its producer command and verifies its recorded identity before showing results. It never chooses the newest run automatically. Alternatively, put `AAAI_ITERATION02_RUN_DIR` in the repository root `.env`. Do not create another environment file. Python-dotenv calls the real functions `load_dotenv` and `find_dotenv`; the aliases below implement the requested `load_env(find_env())` spelling without printing credentials.

Python 3.12 and the notebook, plotting, data and model dependencies are declared in the existing `pyproject.toml`. Select its kernel. This notebook does not depend on cells from another notebook. A full experiment fits models through the recorded producer command; this tutorial loads those real fitted outputs, checks them and teaches how to interpret them. Missing required outputs stop execution instead of triggering synthetic examples.

A small development run has its own protocol identity. To inspect it deliberately, also set `AAAI_ITERATION02_ALLOW_DEVELOPMENT=1`. Its outputs remain labeled incomplete. Do not use that option to bypass a failed evidence verifier.

The experiment is exploratory because the available sources informed earlier work. A dataset folder label is not a diagnosis independently established here. The focal endpoint is normal versus Parkinsons, stroke, cerebral palsy and myopathic labels. The mandatory broad endpoint uses all non-normal folders. The word “normal” refers to the dataset annotation, not a clinical clearance.

In [ ]:
from pathlib import Path
import os, sys, json
from dotenv import load_dotenv as load_env, find_dotenv as find_env

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ambient").is_dir() and (p / "data/gavd").is_dir())
os.chdir(ROOT)
load_env(find_env())
BASE = ROOT / "experiments/aaai"
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from aaai_gait.iteration02_notebook_support import (
    EvidenceRun, package_versions, show_diagram, show_video, show_status,
    optional_evidence, joint_table, source_role_table, plot_counts,
    prediction_example, example_losses, plot_prediction, plot_adequacy,
    plot_timing, plot_source_scores, within_fold_metrics, plot_metric_table,
    fold_json_records, fold_tables, cohort_table, mask_bank_summary,
    measured_window, plot_paired_differences, timing_video_example, external_replication, plot_condition_confusions,
)

run = EvidenceRun.from_env()
display(run.describe().T.rename(columns={0: "value"}))
display(package_versions())
if run.mode not in {"full", "full_study"}:
    display(Markdown("**DEVELOPMENT RUN. This is incomplete evidence and cannot supply final manuscript results.**"))

## 1. Define the comparison before reading its result

Imagine hiding four positions of the left ankle. The visible left knee, hip and foot can already constrain that ankle through local geometry. A same-joint predictor tests temporal interpolation. An own-leg predictor adds local geometry. Only then does the opposite leg provide the added information of interest.

We denote permitted own-leg observations by O, opposite-leg observations by C, and the hidden coordinate patch by Y. All conditions use the same Y, weights, validity and declared body reference. They cannot receive a cached embedding that has already attended to hidden or opposite-leg coordinates.

In [ ]:
show_diagram("residual_predictor")

## 2. Retain the competent baseline and match extra capacity

First fit the normal-only local predictor f(O). Freeze it. The own-leg correction h(O) adds capacity without adding the other leg. The both-leg correction g(O,C) gets the extra context. Both corrections can be zero, preserving the original prediction.

The primary benefit is loss(f+h) minus loss(f+g), not simply loss(f) minus loss(f+g). Otherwise, more parameters could masquerade as cross-leg information. Matching parameter count alone does not equalize effective rank or inductive bias, so the implementation records those limitations. Zero correction is a function-class guarantee, not a guarantee that fitting improves held-out prediction.

## 3. Understand the three motion-independent masks

A 32-frame sequence contains eight four-frame time patches. With 16 joints this gives 128 tokens. U hides 64 distinct tokens uniformly. J chooses one joint uniformly, hides its full eight-token trajectory, then uniformly fills 56 other positions. A hides an eight-token anatomical core, then uses the same fill.

A chooses a group with probability equal to its width divided by 16. Four-joint groups cover two consecutive time patches; two-joint groups cover four. Aligned starts partition the eight-patch window without wrapping. This gives every token the same one-half marginal masking probability as U and J. The policies differ in what disappears together, not their total budget or first-order joint frequency.

In [ ]:
show_diagram("mask_families")
groups = pd.DataFrame([
    ("Torso", [11, 12, 23, 24], 4, 2, [0, 2, 4, 6]),
    ("Left arm", [13, 15], 2, 4, [0, 4]),
    ("Right arm", [14, 16], 2, 4, [0, 4]),
    ("Left distal leg", [25, 27, 29, 31], 4, 2, [0, 2, 4, 6]),
    ("Right distal leg", [26, 28, 30, 32], 4, 2, [0, 2, 4, 6]),
], columns=["group", "indices", "width", "time_patches", "starts"])
groups["selection_probability"] = groups.width / 16
display(groups)
core_probability = (groups.width / 16) * (groups.time_patches / 8)
assert np.allclose(core_probability, 1 / 16)
assert np.isclose(1 / 16 + (15 / 16) * (56 / 120), .5)
print("Analytical construction check only; this is not a learned gait result.")
display(mask_bank_summary(run))
print("A finite 16-draw bank need not have exact one-half empirical token frequencies.")

## 4. Respect the prior art and isolate the training factor

[MAMP](https://arxiv.org/abs/2308.07092) and the original [skeleton S-JEPA](https://www.ecva.net/papers/eccv_2024/papers_ECCV/papers/04755.pdf) use motion-aware masking. Their stochastic sampling is not uniformly random. [SLiM, version 4](https://arxiv.org/html/2603.10648v4) already uses anatomical tubes to reduce easy interpolation, so this project does not invent body-part masking. Action-recognition results do not determine the best gait mask.

The correction-mask experiment keeps one U-trained f fixed while training h and g under U, J or A. A separate JEPA pretraining-mask experiment, if executed, keeps downstream head masks fixed to U. Changing both at once would mix two explanations. Dedicated random streams keep the new sampler from changing source order, target selection or initialization. Both VICReg views must use the assigned family. VICReg can discourage collapse; it does not force disease categories apart.

In [ ]:
display(pd.json_normalize(fold_json_records(run, "selection.json")))
optional_evidence(run, "fixtures.json", "known-dependence, independence, access and zero-correction checks")
show_status(run, "baseline_status.json")

## 5. Inspect one actual predicted normal patch

The next cell loads a deterministic saved normal example from the U policy and reference evaluation bank. It does not select the largest benefit. Each line predicts the same four observed coordinate pairs. The target values come from the real extracted pose; the other lines come from fitted models recorded by the run.

One attractive plot cannot establish accuracy. Look for obvious scale, target-order or coordinate-sign mistakes, then inspect the full normal-development summary. A very accurate local predictor may leave little headroom for any additional context.

In [ ]:
example_row, example_arrays = prediction_example(run)
display(pd.DataFrame([example_row]).drop(columns=[name for name in ("Y", "target", "f", "fh", "fg") if name in example_row], errors="ignore"))
display(plot_prediction(run)); plt.close("all")
display(example_losses(run))
if {"source_id", "sequence_id", "start_seconds", "end_seconds"}.issubset(example_row.index):
    show_video(run, example_row)
else:
    display(Markdown("This saved prediction lacks an exact window key. Its sequence is identified above, but no other interval was substituted for its video."))

## 6. Require normal prediction competence before interpreting an anomaly

The fitted both-leg model must recover useful dependence in known-answer fixtures and meet the declared normal-development gate relative to both f and f+h. Inspect source weighting, effective valid-target exposure and residual magnitude. A J tube removes all history of its selected joint, so the old own-joint eligibility rule cannot silently be used to discard those hard targets.

Draw masks before observation filtering. Confidence and motion must not choose easier masks. A failed gate stays in the report and sets the fold's relational component to zero under the frozen fallback. It does not authorize another architecture search on abnormal AUROC. A null result with almost no valid context is a support limitation rather than evidence that human gait lacks coordination.

In [ ]:
adequacy = run.table("adequacy.csv")
display(adequacy)
display(plot_adequacy(run)); plt.close("all")
display(pd.DataFrame(run.json("normal_test_benefit.json")))
print("Development adequacy and held-out normal benefit are separate evidence. Read each evaluation bank and its conditional interval.")
display(pd.json_normalize(fold_json_records(run, "fitting_record.json")))
print("Ridge fitting is closed form. The separate JEPA probe records its own optimization trace.")

## 7. State the narrow contribution the experiment could support

Useful evidence would show that a controlled training mask improves added-leg prediction on the unchanged U task and that the resulting score adds value beyond simple movement descriptors. Improvement only on a distal-leg challenge bank is task specialization. It is still informative, but it is a different claim.

These masks remain independent of motion intensity. Stochastic positional embeddings, stochastic latent variables and learned mask selectors are separate research axes. They should not be introduced into this comparison after reading a favorable or unfavorable result.

## Preserve the evidence trail

The run directory is the unit of reproducibility. Keep its input hashes, source roles, fitted models, score references, target predictions, source scores, verification record and failed-comparison ledger together. A successful notebook execution establishes that this tutorial can reproduce its displayed analysis. It does not establish a neurological cause or a clinically valid classifier.

The [iteration02 protocol](docs/iteration02/protocol.md), [results](docs/iteration02/results.md), [limitations](docs/iteration02/limitations.md), and [draft02 readiness review](docs/draft02/README.md) must agree with the same run. The earlier draft01 and notebooks 00–11 remain historical evidence. New figures are written as SVG and PDF masters with PNG previews and an input-hash sidecar under this run's `figures/notebooks` directory.

In [ ]:
record_path = run.record('13_own_leg_and_opposite_leg_prediction')
print("Recorded the exact real artifacts read by this notebook:", record_path)
print("This notebook did not replace the experiment producer or alter its source roles.")